# Storm forecasting: local training run (Colab + BigQuery sandbox)

This notebook runs the **training stage** of the project outside Vertex AI and prints the metrics:

1. Copies NOAA Storm Events from the BigQuery public dataset into your sandbox project (same SQL as the pipeline).
2. Builds the weekly feature table (same SQL as the pipeline).
3. Splits by time: last 52 weeks = test, previous 52 = validation, rest = training.
4. Trains the 4 models (Linear Regression, Ridge, Random Forest, XGBoost) with the **prod** hyperparameters.
5. Prints the leaderboard, picks the best of the 4, and runs the **more-than-2% Linear Regression check**.
6. Adds reference forecasts (last week, same week in past years), errors by state and month, and XGBoost feature importance.

**Not included** (these need Vertex AI): the Model Registry, the champion check, schedules and batch inference.

**Before you start:** create a free BigQuery sandbox project (no credit card) and note its **project ID**.
See `notebooks/README.md` for the steps. Then run the cells from top to bottom.

## 1. Settings
Set `PROJECT_ID` to your sandbox project ID. Everything else can stay as it is.

In [ ]:
PROJECT_ID = "your-sandbox-project-id"   # <-- change this
DATASET = "storm_ml"                     # created in your project if missing
LOCATION = "US"                          # must be US: the public NOAA dataset lives there

START_YEAR = 2000                        # same as configs/prod.yaml
END_YEAR = 0                             # 0 = current year
VAL_WEEKS = 52
TEST_WEEKS = 52
MIN_IMPROVEMENT_OVER_BASELINE_PCT = 2.0  # the "beat Linear Regression by more than 2%" rule

# Where to get the project code from:
#   "github" -> clone the repo (works once the code is pushed)
#   "upload" -> upload a .zip of the ml_demo folder when prompted
CODE_SOURCE = "github"
REPO_URL = "https://github.com/prachi615/ml_demo.git"

## 2. Get the project code
The notebook imports the project's own SQL, feature list, models and metrics, so results match the pipeline.

In [ ]:
import os, sys, zipfile, pathlib

CODE_DIR = pathlib.Path("/content/ml_demo") if os.path.isdir("/content") else pathlib.Path.cwd().parent

if not (CODE_DIR / "src" / "storm_prediction").exists():
    if CODE_SOURCE == "github":
        !git clone -q {REPO_URL} {CODE_DIR}
    else:
        from google.colab import files
        uploaded = files.upload()                      # choose ml_demo.zip
        zip_name = next(iter(uploaded))
        with zipfile.ZipFile(zip_name) as zf:
            zf.extractall("/content/_upload")
        # accept a zip that contains the folder itself or just its contents
        root = next(p.parent.parent for p in pathlib.Path("/content/_upload").rglob("storm_prediction/__init__.py"))
        CODE_DIR = root.parent if root.name == "src" else root

sys.path.insert(0, str(CODE_DIR / "src"))
print("Using project code from:", CODE_DIR)

## 3. Install missing libraries
Colab already has most of them (pandas, scikit-learn, BigQuery client).

In [ ]:
%pip install -q xgboost db-dtypes tabulate pyyaml google-cloud-bigquery google-cloud-bigquery-storage

## 4. Log in to Google
In Colab a pop-up asks you to allow access. On a laptop, run `gcloud auth application-default login` in a terminal instead.

In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab")
except ImportError:
    print("Not in Colab: using your gcloud application-default credentials")

## 5. Connect to BigQuery and create the dataset

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
client.create_dataset(bigquery.Dataset(f"{PROJECT_ID}.{DATASET}"), exists_ok=True)
print(f"Dataset ready: {PROJECT_ID}.{DATASET}")

## 6. Check the public dataset has the columns the SQL needs
This is the first time the SQL meets the live dataset, so it checks the schema before running anything.

In [ ]:
needed = {
    "event_id", "episode_id", "state", "state_fips_code", "event_type", "event_begin_time",
    "event_end_time", "injuries_direct", "injuries_indirect", "deaths_direct", "deaths_indirect",
    "damage_property", "damage_crops", "magnitude", "magnitude_type", "event_latitude", "event_longitude",
}
cols = client.query("""
    SELECT column_name, data_type
    FROM `bigquery-public-data.noaa_historic_severe_storms.INFORMATION_SCHEMA.COLUMNS`
    WHERE table_name = 'storms_2020'
""").to_dataframe()
missing = needed - set(cols["column_name"])
assert not missing, f"Public dataset is missing columns {missing}; the SQL needs updating"
years = client.query("""
    SELECT MIN(table_name) AS first_table, MAX(table_name) AS last_table
    FROM `bigquery-public-data.noaa_historic_severe_storms.INFORMATION_SCHEMA.TABLES`
    WHERE table_name LIKE 'storms_%'
""").to_dataframe()
print("Schema OK. Yearly tables available:", years.iloc[0].to_dict())

## 7. Ingest raw events and build features (same SQL as the pipeline)

**Sandbox detail:** the sandbox expires table *partitions* older than 60 days, and every partition here is years old.
A partitioned table would be emptied right after it is written, so the notebook removes the `PARTITION BY` line before running.
Tables themselves expire after 60 days in the sandbox, which is fine for this purpose.

In [ ]:
import re
from datetime import datetime, timezone

from storm_prediction.gcp import render_sql


def sandbox_safe(sql: str) -> str:
    """Drop the PARTITION BY clause (see the note above)."""
    return re.sub(r"^\s*PARTITION BY .*\n", "", sql, flags=re.M)


end_year = END_YEAR or datetime.now(timezone.utc).year
params = dict(project=PROJECT_ID, dataset=DATASET, raw_table="storm_events_raw",
              features_table="storm_features_weekly", start_year=START_YEAR, end_year=end_year)

for name in ("ingest_raw.sql", "build_features.sql"):
    job = client.query(sandbox_safe(render_sql(name, **params)))
    job.result()
    print(f"{name}: done, {job.total_bytes_processed / 1e9:.2f} GB processed")

summary = client.query(f"""
    SELECT COUNT(*) AS raw_events, MIN(event_date) AS first_event, MAX(event_date) AS last_event,
           COUNT(DISTINCT state) AS states
    FROM `{PROJECT_ID}.{DATASET}.storm_events_raw`
""").to_dataframe()
summary

## 8. Load the labelled feature rows

In [ ]:
df = client.query(f"""
    SELECT * FROM `{PROJECT_ID}.{DATASET}.storm_features_weekly`
    WHERE NOT is_forecast_row AND event_count IS NOT NULL
""").to_dataframe()
print(f"{len(df):,} state-weeks, {df['state'].nunique()} states, "
      f"{df['week_start'].min()} to {df['week_start'].max()}")

Quick look at the target: most weeks are small, a few are huge (which is why the models train on log1p).

In [ ]:
df["event_count"].describe(percentiles=[0.5, 0.9, 0.99]).round(2)

## 9. Split by time

In [ ]:
from storm_prediction.data import time_split

train_df, val_df, test_df, split = time_split(df, val_weeks=VAL_WEEKS, test_weeks=TEST_WEEKS)
print(f"train: {len(train_df):,} rows up to {split.train_end.date()}")
print(f"val:   {len(val_df):,} rows up to {split.val_end.date()}")
print(f"test:  {len(test_df):,} rows up to {split.test_end.date()}")

## 10. Train the 4 models (prod hyperparameters from `configs/prod.yaml`)

In [ ]:
import time
import yaml

from storm_prediction.features import split_xy
from storm_prediction.metrics import regression_metrics
from storm_prediction.models import MODEL_TYPES, build_model, predict_counts

with open(CODE_DIR / "configs" / "prod.yaml") as fh:
    HYPERPARAMETERS = yaml.safe_load(fh)["training"]["hyperparameters"]

X_train, y_train = split_xy(train_df)
X_val, y_val = split_xy(val_df)
X_test, y_test = split_xy(test_df)

models, results = {}, []
for model_type in MODEL_TYPES:
    started = time.time()
    model = build_model(model_type, HYPERPARAMETERS.get(model_type, {})).fit(X_train, y_train)
    val_m = regression_metrics(y_val, predict_counts(model, X_val))
    test_m = regression_metrics(y_test, predict_counts(model, X_test))
    models[model_type] = model
    results.append({"model_type": model_type, "train_seconds": round(time.time() - started, 1),
                    **{f"val_{k}": v for k, v in val_m.items() if k != "n_rows"},
                    **{f"test_{k}": v for k, v in test_m.items() if k != "n_rows"}})
    print(f"{model_type:18s} val RMSE {val_m['rmse']:.3f}   test RMSE {test_m['rmse']:.3f}")

## 11. Leaderboard, best of 4, and the Linear Regression check

In [ ]:
import pandas as pd

from storm_prediction.metrics import beats_baseline

board = pd.DataFrame(results).sort_values("val_rmse").reset_index(drop=True)
board.insert(0, "rank", board.index + 1)

best = board.iloc[0]
lr_test_rmse = float(board.loc[board["model_type"] == "linear_regression", "test_rmse"].iloc[0])
passed, reason = beats_baseline(float(best["test_rmse"]), lr_test_rmse, MIN_IMPROVEMENT_OVER_BASELINE_PCT)

cols = ["rank", "model_type", "val_rmse", "val_mae", "test_rmse", "test_mae", "test_r2", "test_wape", "test_bias"]
display(board[cols].round(4))
print(f"\nBest of 4 (lowest validation RMSE): {best['model_type']}")
print(f"Linear Regression check: {'PASSED' if passed else 'FAILED'} - {reason}")
print("In the pipeline this model would now be compared with the current champion."
      if passed else "In the pipeline nothing would be registered this month.")

## 12. Reference forecasts (no model)

For seasonal data, a fair question is "does the model beat simply copying the past?".
These need no training, use the same test weeks, and make good interview talking points.

In [ ]:
references = {
    "last week (lag_1)": test_df["lag_1"],
    "same week last year (lag_52)": test_df["lag_52"],
    "same week, 3-year average": test_df["same_week_mean_3y"],
    "trailing 52-week average": test_df["rolling_mean_52"],
}
ref_rows = [{"forecast": name, **regression_metrics(y_test, pred.astype(float).fillna(0))}
            for name, pred in references.items()]
ref_rows.append({"forecast": f"BEST MODEL: {best['model_type']}",
                 **regression_metrics(y_test, predict_counts(models[best["model_type"]], X_test))})
display(pd.DataFrame(ref_rows)[["forecast", "rmse", "mae", "r2", "wape"]].round(4))

## 13. Where the best model does well and badly

In [ ]:
best_pred = predict_counts(models[best["model_type"]], X_test)
errors = test_df[["state", "week_start", "event_count"]].copy()
errors["prediction"] = best_pred
errors["abs_error"] = (errors["prediction"] - errors["event_count"]).abs()
errors["month"] = pd.to_datetime(errors["week_start"]).dt.month

by_state = (errors.groupby("state")
            .agg(actual_per_week=("event_count", "mean"), mae=("abs_error", "mean"))
            .sort_values("mae", ascending=False).round(2))
by_month = (errors.groupby("month")
            .agg(actual_per_week=("event_count", "mean"), mae=("abs_error", "mean")).round(2))

print("Largest errors by state (test weeks):")
display(by_state.head(10))
print("Error by month (test weeks):")
display(by_month)

## 14. Which features matter (XGBoost)

In [ ]:
xgb_pipeline = models["xgboost"].regressor_
names = xgb_pipeline.named_steps["preprocess"].get_feature_names_out()
importance = pd.Series(xgb_pipeline.named_steps["regressor"].feature_importances_, index=names)
# collapse the one-hot state columns into a single "state" entry
importance = importance.groupby(lambda n: "state" if n.startswith("cat__") else n.removeprefix("num__")).sum()
display(importance.sort_values(ascending=False).head(15).round(4).to_frame("importance"))

## 15. Save the results

In [ ]:
board.to_csv("leaderboard.csv", index=False)
pd.DataFrame(ref_rows).to_csv("reference_forecasts.csv", index=False)
by_state.to_csv("errors_by_state.csv")
print("Saved leaderboard.csv, reference_forecasts.csv, errors_by_state.csv")
try:
    from google.colab import files
    for name in ("leaderboard.csv", "reference_forecasts.csv", "errors_by_state.csv"):
        files.download(name)
except ImportError:
    pass

## Notes

- **Cost:** step 7 reads a few GB from the public dataset; the sandbox gives 1 TB of queries free each month.
- **Clean-up:** sandbox tables delete themselves after 60 days, or delete the `storm_ml` dataset in the BigQuery console.
- **Interview use:** quote the test RMSE / MAE of the best model, its improvement over Linear Regression,
  and how it compares with "same week last year". Those are your real results.